In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [4]:
df = pd.read_csv("CLEAN_dataset.csv")
df

,parent_count,files_changed,additions,deletions,total_changes,message_length,commit_hour,commit_day,developer_experience,directories_changed,buggy_label
0,1,1,4,0,4,98,14,1,0,1,0
1,1,11,554,13,567,638,13,1,0,1,0
2,1,2,946,0,946,938,13,1,0,1,0
3,1,7,166,207,373,420,13,1,0,1,0
4,1,2,52,1,53,2442,13,1,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...
9994,1,4,147,79,226,183,8,0,71,1,1
9995,1,15,12,49,61,54,8,0,0,2,1
9996,1,7,519,152,671,549,15,4,72,3,0
9997,1,1,1,1,2,614,14,4,370,1,1


In [5]:

X = df.drop("buggy_label", axis=1)
y = df["buggy_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## **Decision Tree**

In [10]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [11]:
# base model
dt = DecisionTreeClassifier(random_state=42)

In [12]:
# hyperparameter grid
params = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 5, 7, 10, 15, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 3, 5]
}

In [13]:
# grid search with cross validation
grid = GridSearchCV(
    estimator=dt,
    param_grid=params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

In [14]:
# train
grid.fit(X_train, y_train)


GridSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [3, 5, 7, 10, 15, None],
                         'min_samples_leaf': [1, 3, 5],
                         'min_samples_split': [2, 5, 10]},
             scoring='accuracy')

In [15]:
# best model
best_dt = grid.best_estimator_

In [16]:

print("Best Parameters:")
print(grid.best_params_)

print("\nBest Cross-Validation Accuracy:")
print(grid.best_score_)


Best Parameters:
{'criterion': 'entropy', 'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 2}

Best Cross-Validation Accuracy:
0.6422063789868668


In [17]:
# predictions
y_pred = best_dt.predict(X_test)


In [18]:
# evaluation
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("\nFinal Test Results:")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Final Test Results:
Accuracy: 0.653
Precision: 0.6496815286624203
Recall: 0.5489773950484392
F1-score: 0.5950991831971996

Confusion Matrix:
[[796 275]
 [419 510]]

Classification Report:
              precision    recall  f1-score   support

           0       0.66      0.74      0.70      1071
           1       0.65      0.55      0.60       929

    accuracy                           0.65      2000
   macro avg       0.65      0.65      0.65      2000
weighted avg       0.65      0.65      0.65      2000

